# 从零实现一个小卷积网络：猫狗图片分类（只用 NumPy）

> 课程教学示例 · 不依赖 PyTorch / TensorFlow，**核心只需要 `numpy`**

## 你将学到什么

1. 图片怎么变成神经网络能吃的张量（形状、通道、归一化）
2. 卷积、池化、全连接的前向传播到底在算什么
3. 反向传播怎么推、怎么用**数值梯度**验证自己写对了
4. 训练一个小网络：loss 曲线、混淆矩阵、错例分析
5. 为什么会过拟合，数据增强为什么有用

## 为什么不用 PyTorch

| | 本项目 | PyTorch |
|---|---|---|
| 安装体积 | `numpy` 约 15 MB | CPU 轮子约 200 MB，CUDA 版 2 GB+ |
| 中国网络 | `pip install numpy`（配好镜像几秒装完） | 常遇到超时、CUDA 版本不匹配 |
| 适合 | 理解原理、一次跑通 | 做大项目、上生产 |

学完这一课再去看 PyTorch，你会知道 `nn.Conv2d` 背后到底做了什么。

## 运行前

```bash
pip install numpy          # 必需
pip install matplotlib     # 可选：画图
pip install pillow         # 可选：读你自己拍的照片
```

**按顺序执行**，每个 cell 都会打印形状，形状不对就是上一步错了。


In [ ]:
# ============ 第 0 步：检查环境 ============
import os
import sys
import time
import pickle
import tarfile
import hashlib
import urllib.request

import numpy as np

print("Python :", sys.version.split()[0])
print("NumPy  :", np.__version__)

try:
    import matplotlib
    import matplotlib.pyplot as plt

    HAS_MPL = True
except ImportError:
    HAS_MPL = False
    print("提示：pip install matplotlib 之后可以画图")

try:
    from PIL import Image

    HAS_PIL = True
except ImportError:
    HAS_PIL = False


def setup_chinese_font():
    '''找一个能显示中文的字体；找不到返回 None，后面改用英文标签。'''
    from matplotlib import font_manager, rcParams

    candidates = ["Microsoft YaHei", "SimHei", "PingFang SC", "Hiragino Sans GB",
                  "Noto Sans CJK SC", "Source Han Sans SC", "WenQuanYi Micro Hei",
                  "Sarasa Gothic SC", "Droid Sans Fallback"]
    installed = {f.name for f in font_manager.fontManager.ttflist}
    for name in candidates:
        if name in installed:
            rcParams["font.sans-serif"] = [name] + list(rcParams["font.sans-serif"])
            rcParams["axes.unicode_minus"] = False
            return name
    return None


CJK_FONT = setup_chinese_font() if HAS_MPL else None
if HAS_MPL and CJK_FONT:
    print("matplotlib 中文字体:", CJK_FONT)
elif HAS_MPL:
    print("警告：没找到中文字体，图上的文字会自动用英文代替")
    print("      Linux: sudo apt install fonts-noto-cjk")


def L(cn, en):
    '''有中文字体就用中文，没有就用英文，避免图上全是方块。'''
    return cn if CJK_FONT else en


# --- 教学参数 ---------------------------------------------------------------
# FAST_DEMO=True 会用合成小数据 1 个 epoch 跑完，用来 30 秒内验证环境
FAST_DEMO = False

# 数据来源："cifar"（自动下载）| "folder"（你自己的图片）| "toy"（合成图兜底）
DATA_MODE = "cifar"

DATA_DIR = "data"
IMG_SIZE = 32
N_TRAIN_PER_CLASS = 2000
N_VAL_PER_CLASS = 500
BATCH_SIZE = 64
EPOCHS = 6
LR = 1e-3
SEED = 0

if FAST_DEMO:
    DATA_MODE = "toy"
    N_TRAIN_PER_CLASS = 64
    N_VAL_PER_CLASS = 32
    EPOCHS = 1

os.makedirs(DATA_DIR, exist_ok=True)
print("\n数据模式:", DATA_MODE, "| 图片尺寸: %dx%d" % (IMG_SIZE, IMG_SIZE))


---
## 第 1 步：准备数据

神经网络只认数字，所以每张图片都要变成 `(通道, 高, 宽)` 的数组：

```
一张 32x32 彩色图  ->  (3, 32, 32)，取值 0~1
一批 64 张        ->  (64, 3, 32, 32)
```

三种数据来源任选其一（改上面的 `DATA_MODE`）：

| 模式 | 说明 | 需要什么 |
|---|---|---|
| `cifar` | 自动下载 CIFAR-10，只取 **猫(3)** 和 **狗(5)** | 网络，约 163 MB，只需一次 |
| `folder` | 你自己的 `data/cats_dogs/cat/*.jpg` 与 `dog/*.jpg` | Pillow |
| `toy` | 程序合成的"尖耳猫 / 垂耳狗"，没网时兜底 | 什么都不用 |

为什么用 CIFAR-10：一条公开 URL 就能下、32×32 小图、CPU 也训得动，是标准教学集。


In [ ]:
# ============ 1.1 下载工具：进度条 + 断点续传 + MD5 校验 ============
CIFAR_URL = "https://www.cs.toronto.edu/~kriz/cifar-10-python.tar.gz"
CIFAR_MD5 = "c58f30108f718f92721af3b95e74349a"  # 官方 md5，确认没下坏


def download(url, dest, chunk=1 << 20):
    '''下到 dest。上次中断也没关系：会从 .part 文件断点续传。'''
    part = dest + ".part"
    pos = os.path.getsize(part) if os.path.exists(part) else 0
    req = urllib.request.Request(url, headers={"User-Agent": "catdog-course/1.0"})
    if pos:
        req.add_header("Range", "bytes=%d-" % pos)  # 只请求剩余字节
        print("发现未完成的下载，从 %.1f MB 处继续" % (pos / 1e6))
    with urllib.request.urlopen(req, timeout=30) as r, open(part, "ab") as f:
        total = int(r.headers.get("Content-Length", 0) or 0) + pos
        while True:
            block = r.read(chunk)
            if not block:
                break
            f.write(block)
            pos += len(block)
            if total:
                sys.stdout.write("\r  下载中 %5.1f%%  %.1f/%.1f MB"
                                 % (pos * 100.0 / total, pos / 1e6, total / 1e6))
                sys.stdout.flush()
    sys.stdout.write("\n")
    os.replace(part, dest)


def md5_of(path, chunk=1 << 20):
    h = hashlib.md5()
    with open(path, "rb") as f:
        for b in iter(lambda: f.read(chunk), b""):
            h.update(b)
    return h.hexdigest()


In [ ]:
# ============ 1.2 三种数据加载方式 ============
CAT_INDEX, DOG_INDEX = 3, 5  # CIFAR-10 顺序：0飞机 1汽车 2鸟 3猫 4鹿 5狗 ...


def load_cifar_cats_dogs(n_train_per_class, n_val_per_class):
    '''下载并读取 CIFAR-10，只保留猫和狗，标签重新编号为 0=猫 1=狗。'''
    tar_path = os.path.join(DATA_DIR, "cifar-10-python.tar.gz")
    if not os.path.exists(tar_path):
        print("下载 CIFAR-10（约 163 MB，只需一次）...")
        download(CIFAR_URL, tar_path)
    if md5_of(tar_path) != CIFAR_MD5:
        raise RuntimeError("MD5 不匹配，文件可能下坏了，删掉重下：" + tar_path)
    print("MD5 校验通过")

    def read_batch(tf, name):
        with tf.extractfile("cifar-10-batches-py/" + name) as f:
            d = pickle.load(f, encoding="bytes")  # 官方是 py2 pickle，必须指定编码
        x = d[b"data"].reshape(-1, 3, IMG_SIZE, IMG_SIZE).astype(np.float32) / 255.0
        y = np.array(d[b"labels"], dtype=np.int64)
        return x, y

    tr_x, tr_y, va_x, va_y = [], [], [], []
    with tarfile.open(tar_path) as tf:  # 流式读，不用解压到磁盘
        for i in range(1, 6):
            x, y = read_batch(tf, "data_batch_%d" % i)
            tr_x.append(x)
            tr_y.append(y)
        x, y = read_batch(tf, "test_batch")
        va_x.append(x)
        va_y.append(y)

    Xtr, ytr = np.concatenate(tr_x), np.concatenate(tr_y)
    Xva, yva = np.concatenate(va_x), np.concatenate(va_y)

    def subset(X, y, n, seed):
        rng = np.random.default_rng(seed)
        keep = []
        for cls in (CAT_INDEX, DOG_INDEX):
            idx = np.flatnonzero(y == cls)
            rng.shuffle(idx)
            keep.append(idx[:n])
        idx = np.concatenate(keep)
        rng.shuffle(idx)
        return X[idx], (y[idx] == DOG_INDEX).astype(np.int64)

    return subset(Xtr, ytr, n_train_per_class, 0), subset(Xva, yva, n_val_per_class, 1)


def load_image_folder(root, size=IMG_SIZE):
    '''读取 root/cat/*.jpg 与 root/dog/*.jpg，统一缩放到 size。'''
    if not HAS_PIL:
        raise RuntimeError("读自己的图片需要 Pillow：pip install pillow")
    xs, ys = [], []
    for label, name in enumerate(("cat", "dog")):
        folder = os.path.join(root, name)
        if not os.path.isdir(folder):
            raise FileNotFoundError("缺少目录：" + folder)
        for fn in sorted(os.listdir(folder)):
            if not fn.lower().endswith((".jpg", ".jpeg", ".png", ".bmp", ".webp")):
                continue
            with Image.open(os.path.join(folder, fn)) as im:
                im = im.convert("RGB").resize((size, size), Image.LANCZOS)
                xs.append(np.asarray(im, dtype=np.float32) / 255.0)
            ys.append(label)
    X = np.stack(xs).transpose(0, 3, 1, 2)  # (高,宽,通道) -> (通道,高,宽)
    return X, np.array(ys, dtype=np.int64)


def make_toy_faces(kind, n, rng, size=IMG_SIZE):
    '''合成简化版"尖耳猫 / 垂耳狗"，仅用于没网时验证流程。'''
    yy, xx = np.mgrid[0:size, 0:size].astype(np.float32)

    def tri(a, b, c):
        def side(p, q):
            return (p[0] - q[0]) * (yy - q[1]) - (p[1] - q[1]) * (xx - q[0])

        d1, d2, d3 = side(a, b), side(b, c), side(c, a)
        return ~(((d1 < 0) | (d2 < 0) | (d3 < 0)) & ((d1 > 0) | (d2 > 0) | (d3 > 0)))

    out = np.zeros((n, 3, size, size), np.float32)
    for i in range(n):
        img = rng.normal(0.55, 0.05, (size, size, 3)).astype(np.float32)
        cy, cx = size / 2 + rng.uniform(-1.5, 1.5), size / 2 + rng.uniform(-1.5, 1.5)
        r = rng.uniform(7.0, 8.5)
        color = rng.uniform(0.55, 0.9, 3).astype(np.float32)
        img[((yy - cy) ** 2 + (xx - cx) ** 2) <= r * r] = color
        if kind == "cat":
            for sx in (-1, 1):
                img[tri((cx + sx * r * 0.7, cy - r * 1.15),
                        (cx + sx * r * 0.1, cy - r * 0.5),
                        (cx + sx * r * 1.0, cy - r * 0.3))] = color * 0.85
            for sy in (-2, 0, 2):
                img[int(cy + sy), int(cx - r - 1):int(cx - r + 3)] = 0.95
                img[int(cy + sy), int(cx + r - 2):int(cx + r + 2)] = 0.95
        else:
            for sx in (-1, 1):
                ear = (((xx - (cx + sx * r * 0.95)) / (0.45 * r)) ** 2
                       + ((yy - (cy + r * 0.15)) / (0.95 * r)) ** 2) <= 1
                img[ear] = color * 0.75
            img[((xx - cx) ** 2 + (yy - (cy + r * 0.35)) ** 2) <= (0.22 * r) ** 2] = 0.15
        out[i] = np.clip(img, 0, 1).transpose(2, 0, 1)
    return out


In [ ]:
# ============ 1.3 真正加载数据 ============
t_start = time.time()

if DATA_MODE == "cifar":
    (Xtr, ytr), (Xva, yva) = load_cifar_cats_dogs(N_TRAIN_PER_CLASS, N_VAL_PER_CLASS)
elif DATA_MODE == "folder":
    X, y = load_image_folder(os.path.join(DATA_DIR, "cats_dogs"))
    cut = int(len(X) * 0.8)
    Xtr, ytr, Xva, yva = X[:cut], y[:cut], X[cut:], y[cut:]
else:
    rng = np.random.default_rng(SEED)
    Xtr = np.concatenate([make_toy_faces("cat", N_TRAIN_PER_CLASS, rng),
                          make_toy_faces("dog", N_TRAIN_PER_CLASS, rng)])
    ytr = np.repeat([0, 1], N_TRAIN_PER_CLASS)
    Xva = np.concatenate([make_toy_faces("cat", N_VAL_PER_CLASS, rng),
                          make_toy_faces("dog", N_VAL_PER_CLASS, rng)])
    yva = np.repeat([0, 1], N_VAL_PER_CLASS)

print("训练集 X:", Xtr.shape, " y:", ytr.shape)
print("验证集 X:", Xva.shape, " y:", yva.shape)
print("像素范围: %.2f ~ %.2f" % (Xtr.min(), Xtr.max()))
print("训练集里 猫 %d 张 / 狗 %d 张" % ((ytr == 0).sum(), (ytr == 1).sum()))
print("耗时 %.1f 秒" % (time.time() - t_start))


In [ ]:
# ============ 1.4 看看图长什么样 ============
if HAS_MPL:
    rng = np.random.default_rng(0)
    idx = rng.choice(len(Xtr), 8, replace=False)
    fig, axes = plt.subplots(2, 4, figsize=(10, 5))
    for ax, i in zip(axes.ravel(), idx):
        ax.imshow(Xtr[i].transpose(1, 2, 0))  # (通道,高,宽) -> (高,宽,通道)
        ax.set_title(L("猫", "cat") if ytr[i] == 0 else L("狗", "dog"))
        ax.axis("off")
    fig.suptitle(L("训练样本：32x32 放大后依然很糊，这就是分类的难点",
                   "training samples (32x32, upscaled)"))
    plt.tight_layout()
    plt.show()
else:
    print("没有 matplotlib，跳过可视化")


---
## 第 2 步：预处理

两个关键动作：

1. **标准化**：每个通道减均值、除标准差，让数值围绕 0 分布。不做的话训练极慢甚至不收敛。
2. **打乱**：保证每个 batch 猫狗混合，否则模型会学到"前一半都是猫"这种假规律。

> 均值和标准差必须**只用训练集**算，再拿去处理验证集。用验证集算就是偷看答案。


In [ ]:
# ============ 2.1 标准化（统计量只来自训练集）============
mean = Xtr.mean(axis=(0, 2, 3), keepdims=True)  # 每个通道一个数
std = Xtr.std(axis=(0, 2, 3), keepdims=True) + 1e-8

Xtr = (Xtr - mean) / std
Xva = (Xva - mean) / std

print("训练集 通道均值:", np.round(mean.ravel(), 3))
print("训练集 通道标准差:", np.round(std.ravel(), 3))
print("标准化后范围: %.2f ~ %.2f" % (Xtr.min(), Xtr.max()))
print("形状不变:", Xtr.shape)

perm = np.random.default_rng(SEED).permutation(len(Xtr))
Xtr, ytr = Xtr[perm], ytr[perm]
print("打乱后前 10 个标签:", ytr[:10])


def to_image(x):
    '''把标准化后的 (3,H,W) 还原成 0~1 的 (H,W,3)。显示图片前必须做这一步。'''
    img = (x[None, ...] * std + mean)[0]      # 用 (1,C,1,1) 的统计量广播到 (C,H,W)
    return np.clip(img.transpose(1, 2, 0), 0, 1)


print("反标准化自检: 形状", to_image(Xtr[0]).shape, "范围 %.2f~%.2f"
      % (to_image(Xtr[0]).min(), to_image(Xtr[0]).max()))


---
## 第 3 步：卷积到底在算什么

一个 3×3 卷积核就是 9 个数。把它盖在图片的一小块上，对应位置相乘再相加得到一个数；
一格一格滑过去，就得到一张"特征图"。

```
图片块 (3x3)        卷积核 (3x3)          输出一个数
1 0 1              1  0 -1
0 1 0      *       1  0 -1      =      ...
1 0 1              1  0 -1
```

先手算一次，建立直觉，再写代码。


In [ ]:
# ============ 3.1 手算一次卷积（不用任何库）============
image = np.array([[0, 0, 1, 1, 0],
                  [0, 1, 1, 0, 0],
                  [1, 1, 0, 0, 0],
                  [1, 1, 0, 0, 0],
                  [0, 1, 1, 1, 0]], dtype=np.float32)

kernel = np.array([[1, 0, -1],
                   [1, 0, -1],
                   [1, 0, -1]], dtype=np.float32)  # 专门找"竖直边缘"

out = np.zeros((3, 3), np.float32)
for i in range(3):
    for j in range(3):
        out[i, j] = (image[i:i + 3, j:j + 3] * kernel).sum()

print("图片：\n", image)
print("\n卷积核：\n", kernel)
print("\n输出特征图：\n", out)
print("\n数值大的位置，正是原图里'左暗右亮'的竖直边缘。")
print("结论：卷积核 = 一个可学习的小模板，训练就是让这 9 个数自己变好。")


In [ ]:
# ============ 3.2 同一个核作用在真实图片上 ============
if HAS_MPL:
    gray = to_image(Xtr[0]).mean(axis=2)   # 3 通道压成灰度，便于观察
    kernels = [
        (L("竖边缘", "vertical edge"),
         np.array([[1, 0, -1], [1, 0, -1], [1, 0, -1]], np.float32)),
        (L("横边缘", "horizontal edge"),
         np.array([[1, 1, 1], [0, 0, 0], [-1, -1, -1]], np.float32)),
        (L("模糊", "blur"), np.ones((3, 3), np.float32) / 9.0),
    ]
    fig, axes = plt.subplots(1, 4, figsize=(12, 3))
    axes[0].imshow(gray, cmap="gray")
    axes[0].set_title(L("原图（灰度）", "input (gray)"))
    for k, (name, ker) in enumerate(kernels):
        feat = np.zeros((gray.shape[0] - 2, gray.shape[1] - 2), np.float32)
        for i in range(feat.shape[0]):
            for j in range(feat.shape[1]):
                feat[i, j] = (gray[i:i + 3, j:j + 3] * ker).sum()
        axes[k + 1].imshow(feat, cmap="gray")
        axes[k + 1].set_title(name)
    for ax in axes:
        ax.axis("off")
    plt.tight_layout()
    plt.show()


---
## 第 4 步：从零实现各个层

每层都要实现两个方向：

- **forward**：输入 → 输出，同时把反向传播要用的中间结果存进 `cache`
- **backward**：上游梯度 → 本层参数梯度 + 传给下游的梯度

### 4.1 卷积的加速技巧：im2col

朴素卷积要写四层循环，Python 里慢到没法训练。工业界做法是 **im2col**：
把每个 3×3 小块"拉直"成一行，卷积就变成**一次矩阵乘法**。

```
图片 (N,C,H,W) --im2col--> (N*H_out*W_out, C*kh*kw)
                          @  权重 (K, C*kh*kw).T
                          =  输出 (N*H_out*W_out, K) --reshape--> (N,K,H_out,W_out)
```

`numpy` 的矩阵乘法是 C 实现的，比 Python 循环快两个数量级。


In [ ]:
# ============ 4.1 im2col / col2im ============
def im2col(x, kh, kw, stride=1, pad=0):
    '''(N,C,H,W) -> (N*H_out*W_out, C*kh*kw)，每个卷积窗口变成一行'''
    N, C, H, W = x.shape
    H_out = (H + 2 * pad - kh) // stride + 1
    W_out = (W + 2 * pad - kw) // stride + 1
    if pad:
        x = np.pad(x, ((0, 0), (0, 0), (pad, pad), (pad, pad)))
    cols = np.empty((N, C, kh, kw, H_out, W_out), dtype=x.dtype)
    for i in range(kh):
        i_end = i + stride * H_out
        for j in range(kw):
            j_end = j + stride * W_out
            cols[:, :, i, j, :, :] = x[:, :, i:i_end:stride, j:j_end:stride]
    return cols.transpose(0, 4, 5, 1, 2, 3).reshape(N * H_out * W_out, C * kh * kw), H_out, W_out


def col2im(cols, x_shape, kh, kw, stride=1, pad=0):
    '''im2col 的逆操作：把每行梯度加回原位置（重叠处累加）'''
    N, C, H, W = x_shape
    H_out = (H + 2 * pad - kh) // stride + 1
    W_out = (W + 2 * pad - kw) // stride + 1
    cols = cols.reshape(N, H_out, W_out, C, kh, kw).transpose(0, 3, 4, 5, 1, 2)
    buf = np.zeros((N, C, H + 2 * pad, W + 2 * pad), dtype=cols.dtype)
    for i in range(kh):
        for j in range(kw):
            buf[:, :, i:i + stride * H_out:stride,
                       j:j + stride * W_out:stride] += cols[:, :, i, j]
    if pad:
        buf = buf[:, :, pad:pad + H, pad:pad + W]
    return buf


_t = np.random.default_rng(0).normal(size=(2, 3, 8, 8)).astype(np.float32)
_c, _ho, _wo = im2col(_t, 3, 3, 1, 1)
print("im2col:", _t.shape, "->", _c.shape, "  H_out=%d W_out=%d" % (_ho, _wo))
print("col2im:", col2im(_c, _t.shape, 3, 3, 1, 1).shape)
print("行数 = 样本数 2 x 每图窗口 %d = %d；每行 %d 个数（3 通道 x 3x3）"
      % (_ho * _wo, _c.shape[0], _c.shape[1]))


In [ ]:
# ============ 4.2 卷积层 ============
def conv_forward(x, w, b, stride=1, pad=0):
    '''x:(N,C,H,W)  w:(K,C,kh,kw)  b:(K,)  ->  out:(N,K,H_out,W_out) + cache'''
    K, C, kh, kw = w.shape
    w_flat = w.reshape(K, -1)
    cols, H_out, W_out = im2col(x, kh, kw, stride, pad)
    out = cols @ w_flat.T + b                    # (N*Ho*Wo, K)
    out = out.reshape(x.shape[0], H_out, W_out, K).transpose(0, 3, 1, 2)
    return out, (cols, x.shape, w.shape, w_flat, stride, pad)


def conv_backward(dout, cache):
    '''上游梯度 dout -> (dx, dw, db)'''
    cols, x_shape, w_shape, w_flat, stride, pad = cache
    K, C, kh, kw = w_shape
    N, _, H_out, W_out = dout.shape
    dout_flat = dout.transpose(0, 2, 3, 1).reshape(N * H_out * W_out, K)
    db = dout_flat.sum(axis=0)
    dw = (dout_flat.T @ cols).reshape(w_shape)
    dcols = dout_flat @ w_flat
    dx = col2im(dcols, x_shape, kh, kw, stride, pad)
    return dx, dw, db


# 自测
_x = np.random.default_rng(1).normal(size=(2, 3, 8, 8)).astype(np.float32)
_w = np.random.default_rng(2).normal(size=(4, 3, 3, 3)).astype(np.float32) * 0.1
_b = np.zeros(4, np.float32)
_y, _cache = conv_forward(_x, _w, _b, pad=1)
_dx, _dw, _db = conv_backward(np.ones_like(_y), _cache)
print("输出:", _y.shape, "（pad=1 保持尺寸）")
print("dx:", _dx.shape, "dw:", _dw.shape, "db:", _db.shape)


In [ ]:
# ============ 4.3 最大池化（把 2x2 压成 1 个数，只保留最大的）============
def maxpool_forward(x, size=2):
    N, C, H, W = x.shape
    assert H % size == 0 and W % size == 0, "本课里 32/2/2 都能整除"
    H_out, W_out = H // size, W // size
    blocks = x.reshape(N, C, H_out, size, W_out, size).transpose(0, 1, 2, 4, 3, 5)
    blocks = blocks.reshape(N, C, H_out, W_out, size * size)
    idx = blocks.argmax(axis=-1)                 # 记住最大值是窗口里第几个
    out = np.take_along_axis(blocks, idx[..., None], axis=-1)[..., 0]
    return out, (idx, x.shape, size)


def maxpool_backward(dout, cache):
    idx, x_shape, size = cache
    N, C, H, W = x_shape
    H_out, W_out = H // size, W // size
    dblocks = np.zeros((N, C, H_out, W_out, size * size), dtype=dout.dtype)
    np.put_along_axis(dblocks, idx[..., None], dout[..., None], axis=-1)
    dx = dblocks.reshape(N, C, H_out, W_out, size, size).transpose(0, 1, 2, 4, 3, 5)
    return dx.reshape(N, C, H, W)


_x = np.random.default_rng(3).normal(size=(2, 3, 8, 8)).astype(np.float32)
_p, _pc = maxpool_forward(_x, 2)
print("池化:", _x.shape, "->", _p.shape)
print("反传:", maxpool_backward(np.ones_like(_p), _pc).shape)
print("注意：池化不改变通道数，只是把空间尺寸减半")


In [ ]:
# ============ 4.4 全连接层 + 交叉熵损失 ============
def softmax_cross_entropy(logits, y):
    '''logits:(N,K)  y:(N,) 整数标签 -> (loss, dlogits)

    先减最大值再 exp，防止数值溢出（这是一个很常见的面试题）。
    '''
    z = logits - logits.max(axis=1, keepdims=True)
    exp = np.exp(z)
    probs = exp / exp.sum(axis=1, keepdims=True)
    n = len(y)
    loss = -np.log(probs[np.arange(n), y] + 1e-12).mean()
    dlogits = probs.copy()
    dlogits[np.arange(n), y] -= 1.0              # 这个简洁的梯度是 softmax+CE 的经典结果
    return loss, dlogits / n


_logits = np.array([[2.0, 0.5], [0.1, 1.2]], np.float32)
_y = np.array([0, 1])
_loss, _dl = softmax_cross_entropy(_logits, _y)
print("loss = %.4f（完全预测对的极限是 0）" % _loss)
print("dlogits =\n", np.round(_dl, 4))
print("两行和都是 0，这是 softmax 梯度的性质")


In [ ]:
# ============ 4.5 Adam 优化器（比 SGD 好用得多）============
class Adam:
    '''一阶动量 + 二阶动量，能自动调整每个参数的学习率。'''

    def __init__(self, params, lr=1e-3, beta1=0.9, beta2=0.999, eps=1e-8):
        self.lr, self.b1, self.b2, self.eps = lr, beta1, beta2, eps
        self.m = {k: np.zeros_like(v) for k, v in params.items()}
        self.v = {k: np.zeros_like(v) for k, v in params.items()}
        self.t = 0

    def step(self, params, grads):
        self.t += 1
        for k in params:
            g = grads[k]
            self.m[k] = self.b1 * self.m[k] + (1 - self.b1) * g
            self.v[k] = self.b2 * self.v[k] + (1 - self.b2) * (g * g)
            m_hat = self.m[k] / (1 - self.b1 ** self.t)   # 偏差修正
            v_hat = self.v[k] / (1 - self.b2 ** self.t)
            params[k] -= self.lr * m_hat / (np.sqrt(v_hat) + self.eps)


print("Adam 就绪：4 行核心公式，比 SGD 收敛快很多")


---
## 第 5 步：把它们拼成一个小网络

```
输入 (3, 32, 32)
  └─ Conv 3->8, 3x3, pad=1  -> (8, 32, 32)
  └─ ReLU
  └─ MaxPool 2              -> (8, 16, 16)
  └─ Conv 8->16, 3x3, pad=1 -> (16, 16, 16)
  └─ ReLU
  └─ MaxPool 2              -> (16, 8, 8)
  └─ Flatten                -> 1024
  └─ Dense 1024->128 + ReLU
  └─ Dense 128->2           -> 2 个分数（猫/狗）
```

约 13 万个参数，CPU 上几分钟能训完。真实项目会深得多，但结构思路完全一样。


In [ ]:
# ============ 5.1 网络本体 ============
class SmallCNN:
    def __init__(self, seed=0):
        rng = np.random.default_rng(seed)
        # He 初始化：方差 2/fan_in，配合 ReLU 效果好
        self.p = {
            "w1": rng.normal(0, np.sqrt(2 / (3 * 9)), (8, 3, 3, 3)).astype(np.float32),
            "b1": np.zeros(8, np.float32),
            "w2": rng.normal(0, np.sqrt(2 / (8 * 9)), (16, 8, 3, 3)).astype(np.float32),
            "b2": np.zeros(16, np.float32),
            "w3": rng.normal(0, np.sqrt(2 / 1024), (1024, 128)).astype(np.float32),
            "b3": np.zeros(128, np.float32),
            "w4": rng.normal(0, np.sqrt(2 / 128), (128, 2)).astype(np.float32),
            "b4": np.zeros(2, np.float32),
        }

    def forward(self, x):
        c1, self._k1 = conv_forward(x, self.p["w1"], self.p["b1"], pad=1)
        r1 = np.maximum(c1, 0)
        p1, self._k2 = maxpool_forward(r1, 2)
        c2, self._k3 = conv_forward(p1, self.p["w2"], self.p["b2"], pad=1)
        r2 = np.maximum(c2, 0)
        p2, self._k4 = maxpool_forward(r2, 2)
        flat = p2.reshape(len(p2), -1)
        h = flat @ self.p["w3"] + self.p["b3"]
        rh = np.maximum(h, 0)
        logits = rh @ self.p["w4"] + self.p["b4"]
        self._cache = (c1, c2, p2, flat, h, rh)   # 反向传播要用的中间结果
        return logits

    def loss_and_grad(self, x, y):
        '''前向 + 反向，返回 (loss, 参数字典的梯度, logits)'''
        logits = self.forward(x)
        loss, dlogits = softmax_cross_entropy(logits, y)
        c1, c2, p2, flat, h, rh = self._cache

        g = {}
        g["w4"] = rh.T @ dlogits                  # 全连接层梯度
        g["b4"] = dlogits.sum(axis=0)
        dh = (dlogits @ self.p["w4"].T) * (h > 0)  # ReLU 的导数：正数处为 1
        g["w3"] = flat.T @ dh
        g["b3"] = dh.sum(axis=0)

        dp2 = (dh @ self.p["w3"].T).reshape(p2.shape)
        dr2 = maxpool_backward(dp2, self._k4) * (c2 > 0)
        dp1, g["w2"], g["b2"] = conv_backward(dr2, self._k3)
        dr1 = maxpool_backward(dp1, self._k2) * (c1 > 0)
        _, g["w1"], g["b1"] = conv_backward(dr1, self._k1)
        return loss, g, logits

    def n_params(self):
        return sum(v.size for v in self.p.values())


model = SmallCNN(SEED)
print("参数总量：%d（约 %.1f 万）" % (model.n_params(), model.n_params() / 1e4))
for k, v in model.p.items():
    print("  %-3s %-16s %d 个" % (k, str(v.shape), v.size))


In [ ]:
# ============ 5.3 形状自检：喂一个 batch 看输出对不对 ============
_xb, _yb = Xtr[:BATCH_SIZE], ytr[:BATCH_SIZE]
_logits = model.forward(_xb)
print("输入:", _xb.shape, "-> 输出:", _logits.shape, "（每张图 2 个分数）")
_loss, _g, _ = model.loss_and_grad(_xb, _yb)
print("初始 loss: %.4f（二分类随机猜是 ln2 = 0.693）" % _loss)
print("梯度键:", sorted(_g.keys()))
print("每个梯度的形状都应与参数一致:", all(_g[k].shape == model.p[k].shape for k in _g))


---
## 第 6 步：梯度检查（最重要的一步）

反向传播最容易写错而**不会报错**——loss 照样下降，只是下降得莫名其妙。
标准做法是**数值梯度**：给某个参数加一点 ε，看 loss 变化多少，和解析梯度对比。

$$\frac{\partial L}{\partial w} \approx \frac{L(w+\varepsilon) - L(w-\varepsilon)}{2\varepsilon}$$

经验判据：误差在 `1e-9` ~ `1e-3` 之间都算正常（ReLU / 池化的折点会让个别参数偏一点），
如果是 `1e-1` 以上，基本就是那一层的 backward 写错了。


In [ ]:
# ============ 6.1 数值梯度 vs 反向传播 ============
def numeric_grad(f, arr, index, eps=1e-5):
    old = arr[index]
    arr[index] = old + eps
    f_plus = f()
    arr[index] = old - eps
    f_minus = f()
    arr[index] = old
    return (f_plus - f_minus) / (2 * eps)


def biggest_index(g):
    '''挑梯度最大的位置来检查，比随便挑一个更有意义'''
    return np.unravel_index(int(np.abs(g).argmax()), g.shape)


model = SmallCNN(SEED)
# 用 float64 做检查：float32 的舍入误差会掩盖真正的实现错误
model.p = {k: v.astype(np.float64) for k, v in model.p.items()}
xb, yb = Xtr[:4].astype(np.float64), ytr[:4]
loss, grads, _ = model.loss_and_grad(xb, yb)


def loss_fn():
    return model.loss_and_grad(xb, yb)[0]


print("loss = %.6f" % loss)
print("%-18s %15s %15s %10s" % ("参数(梯度最大处)", "数值梯度", "解析梯度", "相对误差"))
print("-" * 62)
worst = 0.0
for name in ("w1", "w2", "w3", "w4", "b1", "b4"):
    idx = biggest_index(grads[name])
    num = numeric_grad(loss_fn, model.p[name], idx)
    ana = grads[name][idx]
    rel = abs(num - ana) / max(1e-12, abs(num) + abs(ana))
    worst = max(worst, rel)
    print("%-18s %15.8f %15.8f %10.2e" % ("%s%s" % (name, idx), num, ana, rel))

print("\n最大相对误差 %.2e" % worst)
if worst < 1e-2:
    print("通过 ✅ 和解析梯度同量级。")
    print("个别参数落在 1e-3 是正常的：ReLU / 最大池化在 0 附近有折点，")
    print("±eps 的窗口一旦跨过折点，数值梯度就会偏一点。")
    print("真正的实现错误会差 1e-1 以上，或者 loss 根本不下降。")
else:
    print("有问题 ❌ 回头检查对应层的 backward：矩阵转置、padding、累加顺序。")


---
## 第 7 步：开始训练

训练循环就四句话：

```python
for xb, yb in batches:
    loss, grads, _ = model.loss_and_grad(xb, yb)
    opt.step(model.p, grads)
```

每个 epoch 结束后在验证集上算一次准确率，观察**训练准确率**和**验证准确率**的差距。


In [ ]:
# ============ 7.1 工具函数 ============
def iterate_batches(X, y, batch, rng, shuffle=True):
    idx = rng.permutation(len(X)) if shuffle else np.arange(len(X))
    for s in range(0, len(idx) - batch + 1, batch):
        sel = idx[s:s + batch]
        yield X[sel], y[sel]


def predict(model, X, batch=256):
    out = []
    for i in range(0, len(X), batch):
        out.append(model.forward(X[i:i + batch]).argmax(axis=1))
    return np.concatenate(out)


def accuracy(model, X, y):
    return float((predict(model, X) == y).mean())


def augment(x, rng, max_shift=2):
    '''数据增强：随机水平翻转 + 随机平移（效果：让模型见过更多"变体"）'''
    out = np.empty_like(x)
    for i in range(len(x)):
        img = x[i][:, :, ::-1] if rng.random() < 0.5 else x[i]
        dy, dx = rng.integers(-max_shift, max_shift + 1, 2)
        out[i] = np.roll(np.roll(img, dy, axis=1), dx, axis=2)
    return out


def train(model, Xtr, ytr, Xva, yva, epochs, batch=BATCH_SIZE, lr=LR,
          seed=SEED, augment_fn=None, verbose=True):
    opt = Adam(model.p, lr)
    rng = np.random.default_rng(seed)
    hist = {"loss": [], "train_acc": [], "val_acc": []}
    for ep in range(1, epochs + 1):
        t0 = time.time()
        losses = []
        for xb, yb in iterate_batches(Xtr, ytr, batch, rng):
            if augment_fn is not None:
                xb = augment_fn(xb, rng)
            loss, grads, _ = model.loss_and_grad(xb, yb)
            opt.step(model.p, grads)
            losses.append(loss)
        tr_acc = accuracy(model, Xtr, ytr)
        va_acc = accuracy(model, Xva, yva)
        hist["loss"].append(float(np.mean(losses)))
        hist["train_acc"].append(tr_acc)
        hist["val_acc"].append(va_acc)
        if verbose:
            print("epoch %2d/%d  loss %.4f  训练 %.3f  验证 %.3f  (%.1fs)"
                  % (ep, epochs, hist["loss"][-1], tr_acc, va_acc, time.time() - t0))
    return hist


print("工具就绪")


In [ ]:
# ============ 7.2 训练（CPU 上也只要几分钟）============
model = SmallCNN(SEED)
t0 = time.time()
hist = train(model, Xtr, ytr, Xva, yva, EPOCHS)
train_seconds = time.time() - t0
print("\n总训练时间 %.1f 秒" % train_seconds)


In [ ]:
# ============ 7.3 画曲线 ============
if HAS_MPL:
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 3.6))
    ax1.plot(hist["loss"], marker="o")
    ax1.set_xlabel("epoch")
    ax1.set_ylabel(L("训练 loss", "train loss"))
    ax1.set_title(L("损失下降说明在学", "loss goes down"))
    ax1.grid(alpha=0.3)

    ax2.plot(hist["train_acc"], marker="o", label=L("训练集", "train"))
    ax2.plot(hist["val_acc"], marker="o", label=L("验证集", "val"))
    ax2.set_xlabel("epoch")
    ax2.set_ylabel(L("准确率", "accuracy"))
    ax2.set_title(L("两条线的差距 = 过拟合程度", "gap = overfitting"))
    ax2.legend()
    ax2.grid(alpha=0.3)
    plt.tight_layout()
    plt.show()
else:
    print("loss:", [round(v, 4) for v in hist["loss"]])
    print("val_acc:", [round(v, 4) for v in hist["val_acc"]])


---
## 第 8 步：评估

只看准确率不够，还要看**错在哪**。混淆矩阵 + 错例可视化能告诉你：
是猫被认成狗多，还是狗被认成猫多？错的是背景复杂的，还是姿势奇怪的？


In [ ]:
# ============ 8.1 总体准确率 + 混淆矩阵 ============
pred_va = predict(model, Xva)
val_acc = float((pred_va == yva).mean())
print("验证集准确率: %.4f  (%d/%d)" % (val_acc, (pred_va == yva).sum(), len(yva)))

cm = np.zeros((2, 2), dtype=int)
for t, p in zip(yva, pred_va):
    cm[t, p] += 1
print("\n混淆矩阵（行=真实，列=预测）")
print("          预测猫  预测狗")
print("真实猫    %5d   %5d" % (cm[0, 0], cm[0, 1]))
print("真实狗    %5d   %5d" % (cm[1, 0], cm[1, 1]))
print("\n猫的召回率: %.3f   狗的召回率: %.3f"
      % (cm[0, 0] / max(1, cm[0].sum()), cm[1, 1] / max(1, cm[1].sum())))


In [ ]:
# ============ 8.2 看 9 个预测（错的重点看）============
if HAS_MPL:
    rng = np.random.default_rng(0)
    idx = rng.choice(len(Xva), 9, replace=False)
    fig, axes = plt.subplots(3, 3, figsize=(8, 8))
    for ax, i in zip(axes.ravel(), idx):
        ax.imshow(to_image(Xva[i]))          # 显示前要还原成 0~1
        ok = pred_va[i] == yva[i]
        ax.set_title("%s→%s %s" % (L("猫", "cat") if yva[i] == 0 else L("狗", "dog"),
                                   L("猫", "cat") if pred_va[i] == 0 else L("狗", "dog"),
                                   "✓" if ok else "✗"),
                     color="green" if ok else "red")
        ax.axis("off")
    plt.tight_layout()
    plt.show()


---
## 第 9 步：改进实验（本课的重点在方法，不在调参）

把下面 `RUN_EXPERIMENT` 改成 `True` 会再训练一次，用**数据增强**对比：

| 实验 | 训练集准确率 | 验证集准确率 | 结论 |
|---|---|---|---|
| 基线 | 见上 | 见上 | |
| + 数据增强 | ? | ? | 你来填 |

**预期**：训练准确率可能略降，验证准确率上升 —— 这就是"泛化"。
其他可以自己试的杠杆：`EPOCHS`、`LR`、卷积核数量、加一层 Dropout、把 `max_shift` 调大。


In [ ]:
# ============ 9.1 数据增强对比实验 ============
RUN_EXPERIMENT = False

if RUN_EXPERIMENT:
    model_aug = SmallCNN(SEED)
    hist_aug = train(model_aug, Xtr, ytr, Xva, yva, EPOCHS,
                     augment_fn=lambda x, r: augment(x, r, max_shift=2))
    print("\n基线    验证准确率 %.4f" % hist["val_acc"][-1])
    print("数据增强 验证准确率 %.4f" % hist_aug["val_acc"][-1])
else:
    print("把 RUN_EXPERIMENT 改成 True 再运行这一格（约多花一次训练时间）")


### 小结：这个网络为什么有效

1. **卷积**提取局部纹理（耳朵、眼睛、毛发），并且**权重共享**——同一个边缘检测器扫全图。
2. **池化**带来平移不变性：猫往左挪 2 像素，特征图几乎不变。
3. **堆叠**两层后，第二层看到的是"由边缘组成的形状"，感受野更大。
4. **交叉熵 + Adam** 把"预测错"变成一个可以求导的数，然后顺着梯度下山。

这套东西加上残差连接、BatchNorm、更深更宽，就是 ResNet。


---
## 第 10 步：换成你自己的照片（进阶）

准备目录结构（放 100 张以上效果才有意义）：

```
data/cats_dogs/
├── cat/  a.jpg b.jpg ...
└── dog/  c.jpg d.jpg ...
```

然后：

```python
DATA_MODE = "folder"   # 改第一格，或直接在这里覆盖
```

重新运行「第 1 步 1.3」到最后的 cell 即可。**注意**：换数据集后要重新算 `mean/std`。

> 关于数据来源：Kaggle 的 Dogs vs Cats 有 25000 张，需要注册下载；
> 也可以让老师打包好发给大家，或直接用自己拍的猫狗照片。


---
## 思考题

1. 把 `IMG_SIZE` 改成 16，准确率会怎么变？为什么？
2. 去掉标准化（不做 `(x-mean)/std`），训练会发生什么？试着只看 2 个 epoch。
3. 池化换成平均池化，结果会变吗？为什么不能同时去掉所有池化？
4. 把 `EPOCHS` 调到 30，训练准确率接近 1.0 时验证准确率如何？画两条线的差距。
5. 现在只有 2 类。要做 10 类（CIFAR-10 全部），哪几行代码需要改？（提示：`w4` 的形状、标签、`softmax`）

## 什么时候应该改用 PyTorch

| 场景 | 建议 |
|---|---|
| 学习原理、写作业、讲一节课 | 就用这个 notebook |
| 要训 ResNet/ViT、要 GPU、要预训练权重 | 直接上 PyTorch |
| 数据 > 10 万张 | 直接上 PyTorch |
| 只是想在 CPU 上跑个小分类器 | 这个 + 调参就够了 |

## 参考

- CIFAR-10 数据集：<https://www.cs.toronto.edu/~kriz/cifar.html>
- CS231n 卷积网络课程笔记（本课的数学推导基本来自这里）
- 《深度学习》花书 第 9 章：卷积网络

---

**下一步**：把这份 notebook 里的 `SmallCNN` 用 PyTorch 重写一遍（约 30 行），
对比两者的训练速度和代码量，你会对"框架到底替我们做了什么"有非常具体的认识。
